# nanochat d16 base model on ~5.4B ClimbMix tokens, single Colab A100

Fork of karpathy/nanochat: https://github.com/michaelp91-dev/nanochat

* Downloads the val shard + 8 train shards, trains the tokenizer, then **starts training while the other shards download** (`--wait-for-shards`).
* Every checkpoint (every 250 steps) is pushed to the private repo `michaelp91-dev/nanochat-d16` in the background (`ckpt/` + `tokenizer/`), keeping only the latest one.
* After a disconnect: **Runtime -> Run all** again. It pulls the latest checkpoint and the same tokenizer from the repo and resumes.
* At the end it waits for the last push and disconnects the runtime. Nothing goes to Google Drive.

Before running: add a Colab secret named `HF_TOKEN` (write access, notebook access on) and pick an A100 runtime.

In [ ]:
# 1. config
import os, glob, re, time, subprocess
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")  # read by huggingface_hub, never printed
os.environ["NANOCHAT_BASE_DIR"] = "/content/nanochat_cache"
os.environ["OMP_NUM_THREADS"] = "1"

MODEL_REPO   = "michaelp91-dev/nanochat-d16"
DEPTH        = 16
TOTAL_BATCH  = 524_288                         # tokens per optimizer step
TARGET_TOKENS = 5_400_000_000
NUM_ITERS    = TARGET_TOKENS // TOTAL_BATCH    # 10,299
NUM_SHARDS   = 170    # ~55M tokens each, ~35% cropped by the BOS best-fit loader -> enough for 5.4B without repeats
SAVE_EVERY   = 250
BASE = os.environ["NANOCHAT_BASE_DIR"]
CKPT_DIR = f"{BASE}/base_checkpoints/d{DEPTH}"
DATA_DIR = f"{BASE}/base_data_climbmix"
UNASSIGN_ON_FAILURE = True   # disconnect (stop paying) if training crashes; checkpoints are already on HF
print(NUM_ITERS, "steps")

In [ ]:
# 2. get the fork and deps (uses Colab's preinstalled PyTorch)
%cd /content
if not os.path.isdir("/content/nanochat"):
    !git clone https://github.com/michaelp91-dev/nanochat.git
else:
    !cd /content/nanochat && git pull --ff-only
%cd /content/nanochat
!pip install -q rustbpe tiktoken pyarrow wandb kernels filelock psutil huggingface_hub
import torch
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
DEVICE_BATCH = 16 if gpu_gb > 60 else 8   # 80GB vs 40GB A100 (grad accumulation keeps the total batch the same)
print(torch.cuda.get_device_name(0), f"{gpu_gb:.0f} GB -> device batch {DEVICE_BATCH}")

In [ ]:
# 3. resume support: pull the latest checkpoint + its tokenizer from the repo (no-op on a fresh start)
!python -m nanochat.hf_hub pull --repo {MODEL_REPO} --checkpoint-dir {CKPT_DIR}
RESUME_STEP = int(subprocess.check_output(["python", "-m", "nanochat.hf_hub", "latest", "--checkpoint-dir", CKPT_DIR], text=True).split()[-1])
print("resume from step:", RESUME_STEP)

In [ ]:
# 4. data: val + first 8 train shards now (enough for the tokenizer), the rest in the background
!python -m nanochat.dataset -n 8
if os.path.exists(f"{DATA_DIR}/DOWNLOAD_DONE"):
    os.remove(f"{DATA_DIR}/DOWNLOAD_DONE")  # more shards are coming
!nohup python -m nanochat.dataset -n {NUM_SHARDS} > /content/data_download.log 2>&1 &
print("background download started, log: /content/data_download.log")

In [ ]:
# 5. tokenizer: train it unless we pulled the one that matches the checkpoint
if not os.path.exists(f"{BASE}/tokenizer/tokenizer.pkl"):
    !python -m scripts.tok_train
    !python -m scripts.tok_eval
else:
    print("using existing tokenizer")

In [ ]:
# 6. train d16 on one GPU. --wait-for-shards trains while shards download, --hf-repo pushes checkpoints in the background
cmd = ["python", "-m", "scripts.base_train", f"--depth={DEPTH}", f"--num-iterations={NUM_ITERS}",
       f"--total-batch-size={TOTAL_BATCH}", f"--device-batch-size={DEVICE_BATCH}",
       "--window-pattern=L",            # A100 has no FA3; SDPA can't do sliding windows efficiently
       f"--eval-tokens={20 * 524288}",  # ~10M val tokens per eval (default 42M is slow on one GPU)
       f"--save-every={SAVE_EVERY}", "--keep-checkpoints=2",
       f"--wait-for-shards={NUM_SHARDS}", f"--hf-repo={MODEL_REPO}", "--run=dummy"]
if RESUME_STEP > 0:
    cmd.append(f"--resume-from-step={RESUME_STEP}")
print(" ".join(cmd))
os.makedirs(CKPT_DIR, exist_ok=True)
with open(f"{CKPT_DIR}/train_stdout.log", "a") as logf:
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
        logf.write(line)
    rc = p.wait()

def last_step():
    return int(subprocess.check_output(["python", "-m", "nanochat.hf_hub", "latest", "--checkpoint-dir", CKPT_DIR], text=True).split()[-1])

if rc != 0 or last_step() != NUM_ITERS:
    print(f"training stopped early (exit code {rc}, last checkpoint {last_step()})")
    !python -m nanochat.hf_hub wait --checkpoint-dir {CKPT_DIR} --repo {MODEL_REPO}
    try:  # keep the console output with the checkpoints
        from huggingface_hub import HfApi
        HfApi().create_repo(MODEL_REPO, private=True, exist_ok=True)
        HfApi().upload_file(path_or_fileobj=f"{CKPT_DIR}/train_stdout.log", path_in_repo="train_stdout.log", repo_id=MODEL_REPO)
    except Exception as e:
        print("could not upload train_stdout.log:", e)
    if UNASSIGN_ON_FAILURE:
        from google.colab import runtime
        runtime.unassign()
    raise RuntimeError("training did not finish; Run all again to resume")

In [ ]:
# 7. wait for the final checkpoint push, upload the console log
!python -m nanochat.hf_hub wait --checkpoint-dir {CKPT_DIR} --repo {MODEL_REPO}
from huggingface_hub import HfApi
HfApi().upload_file(path_or_fileobj=f"{CKPT_DIR}/train_stdout.log", path_in_repo="train_stdout.log", repo_id=MODEL_REPO)
print(f"done: https://huggingface.co/{MODEL_REPO} (final checkpoint in ckpt/, tokenizer in tokenizer/)")

In [ ]:
# 8. disconnect so the runtime stops using compute units
from google.colab import runtime
runtime.unassign()